In [1]:
# python 3.11.5

import numpy as np
import json
import pandas as pd
import time
import matplotlib.pyplot as plt
from keras import backend as K
from src.learner import *

samples = np.genfromtxt('data/samples.csv', delimiter=",").astype(bool)
samples_consistent = np.genfromtxt('data/samples_consistent.csv', delimiter=",").astype(bool) # see body-rime.Rmd for details
holdouts = np.genfromtxt('data/holdouts.csv', delimiter=",").astype(bool)
tests = np.genfromtxt('data/tests.csv', delimiter=",").astype(bool)

2026-10-27 09:05:43.047713: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


The final hyperparameters we settled on were:

Learning rate: 0.01  
Batch size: 16  
Epochs: 50  

These produced a training and testing accuracy of 0.997 and 0.986, respectively, on the 100 hidden unit model and were associated with a runtime of under one second. While slightly higher train and test accuracies could have been achieved with more advanced tuning and training (e.g. training on more epochs) this configuration allowed us to achieve near-perfect performance on the training and holdout sets and be able to train within reasonable time horizons.  

We will decrease the number of hidden units by 10 succesively: 40, 30, and 20.


In [2]:
with open('data/params.json', 'r') as f:
    cfg = json.load(f)

## Inputs and outputs

In [3]:
X = np.genfromtxt('data/kidwords/orth-kid.csv', delimiter=",")
Y = np.genfromtxt('data/kidwords/phon-kid.csv', delimiter=",")

words = pd.read_csv('data/kidwords/kidwords.csv', header=None)[0].tolist()

Number of orthographic and phonological units...

In [4]:
print(X.shape[1], "orthographic units")
print(Y.shape[1], "phonological units")

260 orthographic units
175 phonological units


## 20 Hidden Units

In [ ]:
hidden = 20
PATH = 'outputs/simulations/'

for sample in range(samples.shape[1]):
    start = time.time()
    sfn = 'sample_' + str(sample) + '_hidden_' + str(hidden) + '_summary.csv'
    pfn = 'sample_' + str(sample) + '_hidden_' + str(hidden) + '_preds.npy'
    afn = 'sample_' + str(sample) + '_hidden_' + str(hidden) + '_accuracies.npy'
    efn = 'sample_' + str(sample) + '_hidden_' + str(hidden) + '_error.npy'


    with open(PATH + sfn, 'w') as f:
        f.write("{},{},{},{},{},{},{},{},{},{},{},{},{}\n".format(
                                                "hidden_units",
                                                "learning_rate",
                                                "batch_size",
                                                "epochs",
                                                "loss_train",
                                                "accuracy_train",
                                                "mse_train",
                                                "loss_test",
                                                "accuracy_test",
                                                "mse_test",
                                                "loss_holdout",
                                                "accuracy_holdout",
                                                "mse_holdout"))

        model = learner(X, Y, cfg['seed'], hidden, optimizer=Adam(learning_rate=cfg['learning_rate']))
        model.fit(X[samples[:, sample]], Y[samples[:, sample]], epochs=cfg['epochs'], batch_size=cfg['batch_size'], verbose=False)

        loss_train, accuracy_train, mse_train = model.evaluate(X[samples[:, sample]], Y[samples[:, sample]], verbose=0) 
        loss_test, accuracy_test, mse_test = model.evaluate(X[tests[:, sample]], Y[tests[:, sample]], verbose=0) 
        loss_holdout, accuracy_holdout, mse_holdout = model.evaluate(X[holdouts[:, sample]], Y[holdouts[:, sample]], verbose=0) 

        preds = model.predict(X)
        np.save(PATH + pfn, preds)

        mse = K.eval(K.mean(K.square(preds - Y), axis = 1))
        np.save(PATH + efn, mse)

        f.write("{},{},{},{},{},{},{},{},{},{},{},{},{}\n".format(
                                                    hidden,
                                                    cfg['learning_rate'],
                                                    cfg['batch_size'],
                                                    cfg['epochs'],
                                                    loss_train,
                                                    accuracy_train,
                                                    mse_train,
                                                    loss_test,
                                                    accuracy_test,
                                                    mse_test,
                                                    loss_holdout,
                                                    accuracy_holdout,
                                                    mse_holdout))
        print(sample, "of", samples.shape[1], "done for HU:", hidden)
        end = time.time()
        print(round(end-start, 4), "seconds elapsed this fit")

## 30 Hidden Units

In [ ]:
hidden = 30
PATH = 'outputs/simulations/'

for sample in range(samples.shape[1]):
    start = time.time()
    sfn = 'sample_' + str(sample) + '_hidden_' + str(hidden) + '_summary.csv'
    pfn = 'sample_' + str(sample) + '_hidden_' + str(hidden) + '_preds.npy'
    afn = 'sample_' + str(sample) + '_hidden_' + str(hidden) + '_accuracies.npy'
    efn = 'sample_' + str(sample) + '_hidden_' + str(hidden) + '_error.npy'


    with open(PATH + sfn, 'w') as f:
        f.write("{},{},{},{},{},{},{},{},{},{},{},{},{}\n".format(
                                                "hidden_units",
                                                "learning_rate",
                                                "batch_size",
                                                "epochs",
                                                "loss_train",
                                                "accuracy_train",
                                                "mse_train",
                                                "loss_test",
                                                "accuracy_test",
                                                "mse_test",
                                                "loss_holdout",
                                                "accuracy_holdout",
                                                "mse_holdout"))

        model = learner(X, Y, cfg['seed'], hidden, optimizer=Adam(learning_rate=cfg['learning_rate']))
        model.fit(X[samples[:, sample]], Y[samples[:, sample]], epochs=cfg['epochs'], batch_size=cfg['batch_size'], verbose=False)

        loss_train, accuracy_train, mse_train = model.evaluate(X[samples[:, sample]], Y[samples[:, sample]], verbose=0) 
        loss_test, accuracy_test, mse_test = model.evaluate(X[tests[:, sample]], Y[tests[:, sample]], verbose=0) 
        loss_holdout, accuracy_holdout, mse_holdout = model.evaluate(X[holdouts[:, sample]], Y[holdouts[:, sample]], verbose=0) 

        preds = model.predict(X)
        np.save(PATH + pfn, preds)

        mse = K.eval(K.mean(K.square(preds - Y), axis = 1))
        np.save(PATH + efn, mse)

        f.write("{},{},{},{},{},{},{},{},{},{},{},{},{}\n".format(
                                                    hidden,
                                                    cfg['learning_rate'],
                                                    cfg['batch_size'],
                                                    cfg['epochs'],
                                                    loss_train,
                                                    accuracy_train,
                                                    mse_train,
                                                    loss_test,
                                                    accuracy_test,
                                                    mse_test,
                                                    loss_holdout,
                                                    accuracy_holdout,
                                                    mse_holdout))
        print(sample, "of", samples.shape[1], "done for HU:", hidden)
        end = time.time()
        print(round(end-start, 4), "seconds elapsed this fit")

## 40 Hidden Units

In [ ]:
hidden = 40
PATH = 'outputs/simulations/'

for sample in range(samples.shape[1]):
    start = time.time()
    sfn = 'sample_' + str(sample) + '_hidden_' + str(hidden) + '_summary.csv'
    pfn = 'sample_' + str(sample) + '_hidden_' + str(hidden) + '_preds.npy'
    afn = 'sample_' + str(sample) + '_hidden_' + str(hidden) + '_accuracies.npy'
    efn = 'sample_' + str(sample) + '_hidden_' + str(hidden) + '_error.npy'


    with open(PATH + sfn, 'w') as f:
        f.write("{},{},{},{},{},{},{},{},{},{},{},{},{}\n".format(
                                                "hidden_units",
                                                "learning_rate",
                                                "batch_size",
                                                "epochs",
                                                "loss_train",
                                                "accuracy_train",
                                                "mse_train",
                                                "loss_test",
                                                "accuracy_test",
                                                "mse_test",
                                                "loss_holdout",
                                                "accuracy_holdout",
                                                "mse_holdout"))

        model = learner(X, Y, cfg['seed'], hidden, optimizer=Adam(learning_rate=cfg['learning_rate']))
        model.fit(X[samples[:, sample]], Y[samples[:, sample]], epochs=cfg['epochs'], batch_size=cfg['batch_size'], verbose=False)

        loss_train, accuracy_train, mse_train = model.evaluate(X[samples[:, sample]], Y[samples[:, sample]], verbose=0) 
        loss_test, accuracy_test, mse_test = model.evaluate(X[tests[:, sample]], Y[tests[:, sample]], verbose=0) 
        loss_holdout, accuracy_holdout, mse_holdout = model.evaluate(X[holdouts[:, sample]], Y[holdouts[:, sample]], verbose=0) 

        preds = model.predict(X)
        np.save(PATH + pfn, preds)

        mse = K.eval(K.mean(K.square(preds - Y), axis = 1))
        np.save(PATH + efn, mse)

        f.write("{},{},{},{},{},{},{},{},{},{},{},{},{}\n".format(
                                                    hidden,
                                                    cfg['learning_rate'],
                                                    cfg['batch_size'],
                                                    cfg['epochs'],
                                                    loss_train,
                                                    accuracy_train,
                                                    mse_train,
                                                    loss_test,
                                                    accuracy_test,
                                                    mse_test,
                                                    loss_holdout,
                                                    accuracy_holdout,
                                                    mse_holdout))
        print(sample, "of", samples.shape[1], "done for HU:", hidden)
        end = time.time()
        print(round(end-start, 4), "seconds elapsed this fit")

# Consistency-based sampling
We will use the same kind of logic for the consistency-based sampling procedure.

## 20 Hidden Units

In [5]:
hidden = 20
PATH = 'outputs/simulations/'

for sample in range(samples_consistent.shape[1]):
    start = time.time()
    sfn = 'consistent_sample_' + str(sample) + '_hidden_' + str(hidden) + '_summary.csv'
    pfn = 'consistent_sample_' + str(sample) + '_hidden_' + str(hidden) + '_preds.npy'
    afn = 'consistent_sample_' + str(sample) + '_hidden_' + str(hidden) + '_accuracies.npy'
    efn = 'consistent_sample_' + str(sample) + '_hidden_' + str(hidden) + '_error.npy'


    with open(PATH + sfn, 'w') as f:
        f.write("{},{},{},{},{},{},{},{},{},{},{},{},{}\n".format(
                                                "hidden_units",
                                                "learning_rate",
                                                "batch_size",
                                                "epochs",
                                                "loss_train",
                                                "accuracy_train",
                                                "mse_train",
                                                "loss_test",
                                                "accuracy_test",
                                                "mse_test",
                                                "loss_holdout",
                                                "accuracy_holdout",
                                                "mse_holdout"))

        model = learner(X, Y, cfg['seed'], hidden, optimizer=Adam(learning_rate=cfg['learning_rate']))
        model.fit(X[samples_consistent[:, sample]], Y[samples_consistent[:, sample]], epochs=cfg['epochs'], batch_size=cfg['batch_size'], verbose=False)

        loss_train, accuracy_train, mse_train = model.evaluate(X[samples_consistent[:, sample]], Y[samples_consistent[:, sample]], verbose=0) 
        loss_test, accuracy_test, mse_test = model.evaluate(X[tests[:, sample]], Y[tests[:, sample]], verbose=0) 
        loss_holdout, accuracy_holdout, mse_holdout = model.evaluate(X[holdouts[:, sample]], Y[holdouts[:, sample]], verbose=0) 

        preds = model.predict(X)
        np.save(PATH + pfn, preds)

        mse = K.eval(K.mean(K.square(preds - Y), axis = 1))
        np.save(PATH + efn, mse)

        f.write("{},{},{},{},{},{},{},{},{},{},{},{},{}\n".format(
                                                    hidden,
                                                    cfg['learning_rate'],
                                                    cfg['batch_size'],
                                                    cfg['epochs'],
                                                    loss_train,
                                                    accuracy_train,
                                                    mse_train,
                                                    loss_test,
                                                    accuracy_test,
                                                    mse_test,
                                                    loss_holdout,
                                                    accuracy_holdout,
                                                    mse_holdout))
        print(sample, "of", samples_consistent.shape[1], "done for HU:", hidden)
        end = time.time()
        print(round(end-start, 4), "seconds elapsed this fit")

2026-10-27 09:06:34.877618: I tensorflow/core/common_runtime/process_util.cc:146] Creating new thread pool with default inter op setting: 2. Tune using inter_op_parallelism_threads for best performance.


90/90 [==============================] - 0s 1ms/step
0 of 10000 done for HU: 20
2.5383 seconds elapsed this fit
90/90 [==============================] - 0s 919us/step
1 of 10000 done for HU: 20
1.8571 seconds elapsed this fit
90/90 [==============================] - 0s 854us/step
2 of 10000 done for HU: 20
1.8612 seconds elapsed this fit
90/90 [==============================] - 0s 886us/step
3 of 10000 done for HU: 20
1.7898 seconds elapsed this fit
90/90 [==============================] - 0s 1ms/step
4 of 10000 done for HU: 20
1.832 seconds elapsed this fit
90/90 [==============================] - 0s 871us/step
5 of 10000 done for HU: 20
2.044 seconds elapsed this fit
90/90 [==============================] - 0s 808us/step
6 of 10000 done for HU: 20
1.6924 seconds elapsed this fit
90/90 [==============================] - 0s 861us/step
7 of 10000 done for HU: 20
1.6896 seconds elapsed this fit
90/90 [==============================] - 0s 879us/step
8 of 10000 done for HU: 20
1.6887 secon

## 30 Hidden Units

In [6]:
hidden = 30
PATH = 'outputs/simulations/'

for sample in range(samples_consistent.shape[1]):
    start = time.time()
    sfn = 'consistent_sample_' + str(sample) + '_hidden_' + str(hidden) + '_summary.csv'
    pfn = 'consistent_sample_' + str(sample) + '_hidden_' + str(hidden) + '_preds.npy'
    afn = 'consistent_sample_' + str(sample) + '_hidden_' + str(hidden) + '_accuracies.npy'
    efn = 'consistent_sample_' + str(sample) + '_hidden_' + str(hidden) + '_error.npy'


    with open(PATH + sfn, 'w') as f:
        f.write("{},{},{},{},{},{},{},{},{},{},{},{},{}\n".format(
                                                "hidden_units",
                                                "learning_rate",
                                                "batch_size",
                                                "epochs",
                                                "loss_train",
                                                "accuracy_train",
                                                "mse_train",
                                                "loss_test",
                                                "accuracy_test",
                                                "mse_test",
                                                "loss_holdout",
                                                "accuracy_holdout",
                                                "mse_holdout"))

        model = learner(X, Y, cfg['seed'], hidden, optimizer=Adam(learning_rate=cfg['learning_rate']))
        model.fit(X[samples_consistent[:, sample]], Y[samples_consistent[:, sample]], epochs=cfg['epochs'], batch_size=cfg['batch_size'], verbose=False)

        loss_train, accuracy_train, mse_train = model.evaluate(X[samples_consistent[:, sample]], Y[samples_consistent[:, sample]], verbose=0) 
        loss_test, accuracy_test, mse_test = model.evaluate(X[tests[:, sample]], Y[tests[:, sample]], verbose=0) 
        loss_holdout, accuracy_holdout, mse_holdout = model.evaluate(X[holdouts[:, sample]], Y[holdouts[:, sample]], verbose=0) 

        preds = model.predict(X)
        np.save(PATH + pfn, preds)

        mse = K.eval(K.mean(K.square(preds - Y), axis = 1))
        np.save(PATH + efn, mse)

        f.write("{},{},{},{},{},{},{},{},{},{},{},{},{}\n".format(
                                                    hidden,
                                                    cfg['learning_rate'],
                                                    cfg['batch_size'],
                                                    cfg['epochs'],
                                                    loss_train,
                                                    accuracy_train,
                                                    mse_train,
                                                    loss_test,
                                                    accuracy_test,
                                                    mse_test,
                                                    loss_holdout,
                                                    accuracy_holdout,
                                                    mse_holdout))
        print(sample, "of", samples_consistent.shape[1], "done for HU:", hidden)
        end = time.time()
        print(round(end-start, 4), "seconds elapsed this fit")

90/90 [==============================] - 0s 991us/step
0 of 10000 done for HU: 30
1.8978 seconds elapsed this fit
90/90 [==============================] - 0s 915us/step
1 of 10000 done for HU: 30
1.9167 seconds elapsed this fit
90/90 [==============================] - 0s 1ms/step
2 of 10000 done for HU: 30
1.9143 seconds elapsed this fit
90/90 [==============================] - 0s 937us/step
3 of 10000 done for HU: 30
1.9048 seconds elapsed this fit
90/90 [==============================] - 0s 933us/step
4 of 10000 done for HU: 30
1.9261 seconds elapsed this fit
90/90 [==============================] - 0s 943us/step
5 of 10000 done for HU: 30
1.8892 seconds elapsed this fit
90/90 [==============================] - 0s 882us/step
6 of 10000 done for HU: 30
1.8681 seconds elapsed this fit
90/90 [==============================] - 0s 943us/step
7 of 10000 done for HU: 30
1.8432 seconds elapsed this fit
90/90 [==============================] - 0s 966us/step
8 of 10000 done for HU: 30
1.8695 s

## 40 Hidden Units

In [7]:
hidden = 40
PATH = 'outputs/simulations/'

for sample in range(samples_consistent.shape[1]):
    start = time.time()
    sfn = 'consistent_sample_' + str(sample) + '_hidden_' + str(hidden) + '_summary.csv'
    pfn = 'consistent_sample_' + str(sample) + '_hidden_' + str(hidden) + '_preds.npy'
    afn = 'consistent_sample_' + str(sample) + '_hidden_' + str(hidden) + '_accuracies.npy'
    efn = 'consistent_sample_' + str(sample) + '_hidden_' + str(hidden) + '_error.npy'


    with open(PATH + sfn, 'w') as f:
        f.write("{},{},{},{},{},{},{},{},{},{},{},{},{}\n".format(
                                                "hidden_units",
                                                "learning_rate",
                                                "batch_size",
                                                "epochs",
                                                "loss_train",
                                                "accuracy_train",
                                                "mse_train",
                                                "loss_test",
                                                "accuracy_test",
                                                "mse_test",
                                                "loss_holdout",
                                                "accuracy_holdout",
                                                "mse_holdout"))

        model = learner(X, Y, cfg['seed'], hidden, optimizer=Adam(learning_rate=cfg['learning_rate']))
        model.fit(X[samples_consistent[:, sample]], Y[samples_consistent[:, sample]], epochs=cfg['epochs'], batch_size=cfg['batch_size'], verbose=False)

        loss_train, accuracy_train, mse_train = model.evaluate(X[samples_consistent[:, sample]], Y[samples_consistent[:, sample]], verbose=0) 
        loss_test, accuracy_test, mse_test = model.evaluate(X[tests[:, sample]], Y[tests[:, sample]], verbose=0) 
        loss_holdout, accuracy_holdout, mse_holdout = model.evaluate(X[holdouts[:, sample]], Y[holdouts[:, sample]], verbose=0) 

        preds = model.predict(X)
        np.save(PATH + pfn, preds)

        mse = K.eval(K.mean(K.square(preds - Y), axis = 1))
        np.save(PATH + efn, mse)

        f.write("{},{},{},{},{},{},{},{},{},{},{},{},{}\n".format(
                                                    hidden,
                                                    cfg['learning_rate'],
                                                    cfg['batch_size'],
                                                    cfg['epochs'],
                                                    loss_train,
                                                    accuracy_train,
                                                    mse_train,
                                                    loss_test,
                                                    accuracy_test,
                                                    mse_test,
                                                    loss_holdout,
                                                    accuracy_holdout,
                                                    mse_holdout))
        print(sample, "of", samples_consistent.shape[1], "done for HU:", hidden)
        end = time.time()
        print(round(end-start, 4), "seconds elapsed this fit")

90/90 [==============================] - 0s 1ms/step
0 of 10000 done for HU: 40
2.0228 seconds elapsed this fit
90/90 [==============================] - 0s 903us/step
1 of 10000 done for HU: 40
1.985 seconds elapsed this fit
90/90 [==============================] - 0s 947us/step
2 of 10000 done for HU: 40
1.977 seconds elapsed this fit
90/90 [==============================] - 0s 907us/step
3 of 10000 done for HU: 40
1.983 seconds elapsed this fit
90/90 [==============================] - 0s 947us/step
4 of 10000 done for HU: 40
1.9695 seconds elapsed this fit
90/90 [==============================] - 0s 874us/step
5 of 10000 done for HU: 40
1.9643 seconds elapsed this fit
90/90 [==============================] - 0s 871us/step
6 of 10000 done for HU: 40
1.9507 seconds elapsed this fit
90/90 [==============================] - 0s 898us/step
7 of 10000 done for HU: 40
1.9678 seconds elapsed this fit
90/90 [==============================] - 0s 880us/step
8 of 10000 done for HU: 40
1.946 secon